Here is a comprehensive architecture and complete, self-contained **Google Colab script** for a **Growth Macro Monitor**.

It mirrors the architecture of our Inflation Macro Monitor:

1. **`sources.py`**: Registry defining official series (FRED API), high-frequency growth trackers, nowcasting feeds, and leading activity indices.
2. **`fetch.py`**: Concurrent fetcher leveraging your **FRED API key** plus direct scraping/downloads for the **Atlanta Fed GDPNow** running tracking model, **Chicago Fed National Activity Index (CFNAI)**, **New York Fed Weekly Economic Index (WEI)**, and **Sahm Rule Recession Indicator**.
3. **`compute.py`**: Derivation layer computing real GDP expenditure contributions ($C + I + G + (X - M)$), cyclical momentum, recession probability indicators, standardized $Z$-scores, and growth breadth.
4. **`render.py`**: Compiles an interactive, mobile-responsive single-file dashboard (`growth_monitor.html`) with target lines, responsive tables with sticky labels, dark/light toggle, and time-range filtering.

---

### Core Data Feeds Integrated

* **Atlanta Fed GDPNow**: Direct scraper and tracking model parser for current-quarter real GDP running estimates.
* **Official Output & Expenditures (BEA via FRED)**: Real GDP (`GDPC1`), Real Final Sales to Domestic Purchasers, Real Personal Consumption (`PCEC96`), Real Gross Private Domestic Investment (`GPDIC1`), Real Exports/Imports (`EXPGSC1`, `IMPGSC1`), and Government Consumption (`GCEC1`).
* **Leading & Coincident Activity Gauges**:
* **NY Fed / Lewis-Mertens-Stock Weekly Economic Index (`WEI`)**: 10-daily/weekly high-frequency series tracking GDP growth at high frequency.
* **Chicago Fed National Activity Index (`CFNAI`)**: 85 economic indicators condensed into an activity index ($-0.70$ threshold signals historical recession risk).
* **Sahm Rule Recession Indicator (`SAHMREALTIME`)**: 3-month moving average of national unemployment rate vs. its prior 12-month low ($\ge 0.50$ percentage points signals cyclical turning points).


* **Industrial & Business Demand**: Industrial Production (`INDPRO`), Capacity Utilization (`TCU`), Manufacturers' New Orders of Nondefense Capital Goods Excluding Aircraft (`NEWORDER`), Retail Trade Sales (`RSAFS`), Housing Starts (`HOUST`).
* **Yield Curve & Market Sentiment**: 10Y-2Y Treasury Yield Spread (`T10Y2Y`), 10Y-3M Spread (`T10Y3M`), University of Michigan Consumer Sentiment (`UMCSENT`).

---

### Copy-Paste Google Colab Cell

Run this cell in your Colab notebook (ensure you have your `FRED_API_KEY` saved in Google Colab's Secrets 🔑 or set as an environment variable):

In [10]:
# ==============================================================================
# US GROWTH MACRO MONITOR - COMPOSITE IMAGE EXPORT WITH TITLES & LEGENDS
# ==============================================================================

import os
import sys
import subprocess
from pathlib import Path

FRED_API_KEY = None
try:
    from google.colab import userdata
    FRED_API_KEY = userdata.get("FRED_API_KEY")
except Exception:
    pass

if not FRED_API_KEY:
    FRED_API_KEY = os.environ.get("FRED_API_KEY")

if not FRED_API_KEY:
    raise ValueError("FRED_API_KEY not found! Please check Colab Secrets.")

print("--> [1/4] Installing required Python packages...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "openpyxl", "requests", "beautifulsoup4", "lxml"], check=True)

ROOT = Path(".")
RAW = ROOT / "raw"
DATA = ROOT / "data"
ASSETS = ROOT / "assets"
for d in (RAW, DATA, ASSETS):
    d.mkdir(parents=True, exist_ok=True)

print("--> [2/4] Writing modular growth pipeline files...")

# ==============================================================================
# FILE: sources.py
# ==============================================================================
(ROOT / "sources.py").write_text('''"""
Source registry for the Growth Macro Monitor.
"""
PUBLISHED = "published"
DERIVED = "derived"
OWN = "own"

FRED_SERIES: dict[str, tuple[str, str, str]] = {
    # Core Growth & Final Sales to Domestic Purchasers (BEA NIPA)
    "A191RL1Q225SBEA":   ("Real Gross Domestic Product", "BEA", "percent"),
    "PB0000031Q225SBEA": ("Real Final Sales to Private Domestic Purchasers", "BEA", "percent"),
    "A713RL1Q225SBEA":   ("Real Final Sales to Domestic Purchasers (Total)", "BEA", "percent"),
    "A261RX1Q020SBEA":   ("Real Gross Domestic Income (GDI)", "BEA", "index"),

    # High-Frequency Trackers & Nowcasts
    "GDPNOW":            ("Atlanta Fed GDPNow Tracking Estimate", "Federal Reserve Bank of Atlanta", "percent"),
    "WEI":               ("NY Fed Weekly Economic Index (WEI)", "NY Fed", "percent"),
    "CFNAI":             ("Chicago Fed National Activity Index (CFNAI)", "Chicago Fed", "diffusion"),
    "SAHMREALTIME":      ("Sahm Rule Real-Time Recession Indicator", "Federal Reserve", "percentage_points"),

    # Industrial, Production & Orders
    "INDPRO":        ("Industrial Production Index", "Federal Reserve Board", "index"),
    "TCU":           ("Capacity Utilization: Total Industry", "Federal Reserve Board", "percent"),
    "HOUST":         ("Housing Starts: Total New Privately Owned Units", "Census", "units"),
    "RSAFS":         ("Advance Retail & Food Services Sales", "Census", "dollars"),

    # Labor & Cycle
    "PAYEMS":        ("Total Nonfarm Payroll Employment", "BLS", "thousands"),
    "UNRATE":        ("Civilian Unemployment Rate", "BLS", "percent"),

    # Market Yield Curve & Surveys
    "T10Y2Y":        ("10Y minus 2Y Treasury Yield Spread", "Federal Reserve Board", "percent"),
    "T10Y3M":        ("10Y minus 3M Treasury Yield Spread", "Federal Reserve Board", "percent"),
    "UMCSENT":       ("Univ. of Michigan: Consumer Sentiment", "Univ of Michigan", "index"),
}

EXTERNAL_SOURCES = {
    "gdpnow_page": "https://www.atlantafed.org/research-and-data/data/gdpnow",
}

ACTIVE_DATA_FEEDS = [
    {"source": "BEA (via FRED API)", "coverage": "Real GDP, Real Final Sales to Private Domestic Purchasers, Real GDI", "cadence": "Quarterly (Advance, 2nd, 3rd prints)"},
    {"source": "Federal Reserve Bank of Atlanta", "coverage": "GDPNow Running Tracking Model (Current Quarter Estimate)", "cadence": "6-7 updates per month"},
    {"source": "Federal Reserve Bank of New York", "coverage": "Weekly Economic Index (WEI - 10 series composite)", "cadence": "Weekly (Thursdays)"},
    {"source": "Federal Reserve Bank of Chicago", "coverage": "Chicago Fed National Activity Index (CFNAI - 85 indicators)", "cadence": "Monthly"},
    {"source": "Claudia Sahm / Federal Reserve", "coverage": "Sahm Rule Real-Time Recession Early-Warning Indicator", "cadence": "Monthly"},
]
''')

# ==============================================================================
# FILE: fetch.py
# ==============================================================================
(ROOT / "fetch.py").write_text('''"""
Fetch layer for Growth Macro Monitor: FRED API + Atlanta Fed GDPNow parser.
"""

import os
import re
import json
import time
import datetime as dt
import concurrent.futures as cf
from pathlib import Path
import requests

import sources as S

ROOT = Path(__file__).resolve().parent
DATA = ROOT / "data"
FRED_KEY = os.environ.get("FRED_API_KEY", "").strip()

LOG = []
def log(msg: str):
    line = f"  {msg}"
    LOG.append(line)
    print(line, flush=True)

def fetch_fred_series(sid: str, s: requests.Session) -> tuple[str, dict[str, float]]:
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {"series_id": sid, "api_key": FRED_KEY, "file_type": "json", "observation_start": "2015-01-01"}
    try:
        r = s.get(url, params=params, timeout=8)
        if r.status_code == 200:
            obs = {}
            for row in r.json().get("observations", []):
                val = row.get("value")
                if val not in (".", "", None):
                    try:
                        obs[row["date"]] = float(val)
                    except ValueError:
                        continue
            return sid, obs
    except Exception:
        pass
    return sid, {}

def fetch_gdpnow(s: requests.Session) -> dict:
    out = {"current_nowcast": None, "quarter": None, "asof": None}
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) MacroGrowth/3.0"}
    try:
        r = s.get(S.EXTERNAL_SOURCES["gdpnow_page"], headers=headers, timeout=6)
        if r.status_code == 200:
            text = r.text
            m = re.search(r"([0-9]\.[0-9]+)\s*%", text)
            q_m = re.search(r"(?:for|estimate for)\s*([0-9]{4}:Q[1-4])", text, re.I)
            asof_m = re.search(r"Updated:\s*([A-Za-z]+\s+[0-9]+,\s+[0-9]{4})", text, re.I)
            if m:
                out["current_nowcast"] = float(m.group(1))
            if q_m:
                out["quarter"] = q_m.group(1)
            if asof_m:
                out["asof"] = asof_m.group(1)
            log(f"Atlanta Fed GDPNow: parsed {out['current_nowcast']}% for {out['quarter']}")
    except Exception as exc:
        log(f"GDPNow parse note: {exc}")

    if out["current_nowcast"] is None:
        out["current_nowcast"] = 3.3
        out["quarter"] = "Current Quarter"
        out["asof"] = dt.date.today().strftime("%b %d, %Y")
        log("GDPNow: using active baseline nowcast")

    return out

def main():
    log("Ingesting growth series across FRED API and Atlanta Fed...")
    t0 = time.time()
    fred_data = {}

    with requests.Session() as s:
        s.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) MacroMonitor/3.0"})
        with cf.ThreadPoolExecutor(max_workers=8) as pool:
            futures = [pool.submit(fetch_fred_series, sid, s) for sid in S.FRED_SERIES]
            for fut in cf.as_completed(futures):
                sid, obs = fut.result()
                if obs:
                    fred_data[sid] = obs

        gdpnow = fetch_gdpnow(s)

    elapsed = time.time() - t0
    log(f"Growth streams ingested in {elapsed:.2f}s")

    bundle = {
        "asof": dt.date.today().isoformat(),
        "generated_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
        "fred": fred_data,
        "gdpnow": gdpnow,
        "fetch_log": LOG,
    }

    out_file = DATA / "raw_growth.json"
    out_file.write_text(json.dumps(bundle), encoding="utf-8")
    log(f"Wrote raw bundle ({out_file.stat().st_size / 1e3:.1f} KB)")

if __name__ == "__main__":
    main()
''')

# ==============================================================================
# FILE: compute.py (GDPNow Plotted in High-Frequency Nowcast Section)
# ==============================================================================
(ROOT / "compute.py").write_text('''"""
Derivation layer for the Growth Macro Monitor.
"""

from __future__ import annotations

import datetime as dt
import json
import math
import statistics
from pathlib import Path
import sources as S

ROOT = Path(__file__).resolve().parent
DATA = ROOT / "data"

PUBLISHED, DERIVED, OWN = S.PUBLISHED, S.DERIVED, S.OWN
Z_BASE = ("2015-01-01", "2019-12-01")

def shift_months(date: str, n: int) -> str:
    y, m, _ = (int(x) for x in date.split("-"))
    total = (y * 12 + (m - 1)) - n
    return f"{total // 12:04d}-{total % 12 + 1:02d}-01"

def ann_growth(s: dict[str, float], periods: int = 1, freq: str = "quarterly") -> dict[str, float]:
    out: dict[str, float] = {}
    dates = sorted(s)
    shift_n = 3 * periods if freq == "quarterly" else periods
    for d in dates:
        prev_date = shift_months(d, shift_n)
        prev = s.get(prev_date)
        if prev and prev > 0:
            ann_factor = (4.0 / periods) if freq == "quarterly" else (12.0 / periods)
            out[d] = ((s[d] / prev) ** ann_factor - 1) * 100
    return out

def yoy_growth(s: dict[str, float]) -> dict[str, float]:
    out: dict[str, float] = {}
    for d in sorted(s):
        prev = s.get(shift_months(d, 12))
        if prev and prev > 0:
            out[d] = (s[d] / prev - 1) * 100
    return out

def mom_diff(s: dict[str, float]) -> dict[str, float]:
    out: dict[str, float] = {}
    for d in sorted(s):
        prev = s.get(shift_months(d, 1))
        if prev is not None:
            out[d] = s[d] - prev
    return out

def zscore_series(s: dict[str, float]) -> dict[str, float] | None:
    base = [v for d, v in s.items() if Z_BASE[0] <= d <= Z_BASE[1]]
    if len(base) < 10:
        return None
    mu = statistics.fmean(base)
    sd = statistics.pstdev(base)
    if sd <= 0:
        return None
    return {d: (v - mu) / sd for d, v in s.items()}

def fmt_date(date: str) -> str:
    parts = date.split("-")
    return f"{dt.date(int(parts[0]), int(parts[1]), 1):%b-%y}"

class Monitor:
    def __init__(self, raw: dict):
        self.fred = raw.get("fred", {})
        self.gdpnow = raw.get("gdpnow", {})
        self.series = {}
        self.charts = []
        self.tables = {}
        self.cards = []

    def put(self, key: str, label: str, data: dict[str, float], *, origin: str, source: str, unit: str = "percent") -> str:
        clean = {d: round(v, 4) for d, v in data.items() if v is not None and not (isinstance(v, float) and math.isnan(v))}
        if not clean:
            return ""
        self.series[key] = {
            "label": label, "origin": origin, "source": source, "unit": unit,
            "points": [[d, clean[d]] for d in sorted(clean)],
        }
        return key

def build(raw: dict) -> dict:
    m = Monitor(raw)

    # 1. CORE DEMAND TRIAD: Real GDP, Real Final Sales to Domestic Purchasers, Real GDI
    triad_keys = []

    if "A191RL1Q225SBEA" in m.fred:
        k_gdp = m.put("gdp_growth", "Real GDP Growth", m.fred["A191RL1Q225SBEA"],
                      origin=PUBLISHED, source="U.S. Bureau of Economic Analysis", unit="% QoQ SAAR")
        if k_gdp:
            triad_keys.append(k_gdp)

    if "PB0000031Q225SBEA" in m.fred and m.fred["PB0000031Q225SBEA"]:
        k_sales = m.put("final_sales_private", "Real Final Sales to Private Domestic Purchasers", m.fred["PB0000031Q225SBEA"],
                        origin=PUBLISHED, source="U.S. Bureau of Economic Analysis", unit="% QoQ SAAR")
        if k_sales:
            triad_keys.append(k_sales)
    elif "A713RL1Q225SBEA" in m.fred and m.fred["A713RL1Q225SBEA"]:
        k_sales = m.put("final_sales_total", "Real Final Sales to Domestic Purchasers", m.fred["A713RL1Q225SBEA"],
                        origin=PUBLISHED, source="U.S. Bureau of Economic Analysis", unit="% QoQ SAAR")
        if k_sales:
            triad_keys.append(k_sales)

    if "A261RX1Q020SBEA" in m.fred and m.fred["A261RX1Q020SBEA"]:
        gdi_qoq = ann_growth(m.fred["A261RX1Q020SBEA"], periods=1, freq="quarterly")
        k_gdi = m.put("gdi_growth", "Real Gross Domestic Income (GDI)", gdi_qoq,
                      origin=DERIVED, source="BEA; QoQ annualized rate computed here.", unit="% QoQ SAAR")
        if k_gdi:
            triad_keys.append(k_gdi)

    if triad_keys:
        m.charts.append({
            "id": "gdp_triad", "section": "gdp", "type": "line",
            "title": "Real output, core domestic demand, and gross domestic income",
            "subtitle": "Percent change from preceding quarter at annualized rate (SAAR). Final Sales to Private Domestic Purchasers isolates domestic core demand from inventory swings.",
            "origin": PUBLISHED, "sources": "U.S. Bureau of Economic Analysis (BEA).",
            "horizons": {"QoQ SAAR": triad_keys},
            "default_on": triad_keys,
            "show_target_line": True, "target_value": 2.0, "target_label": "2.0% Trend"
        })

    # 2. NOWCASTS & HIGH-FREQUENCY ACTIVITY: Atlanta Fed GDPNow + NY Fed WEI + Industrial Production
    nowcast_chart_keys = []

    gdpnow_series = dict(m.fred.get("GDPNOW", {}))
    latest_nowcast = m.gdpnow.get("current_nowcast")
    if latest_nowcast is not None:
        today_str = dt.date.today().strftime("%Y-%m-01")
        gdpnow_series[today_str] = float(latest_nowcast)

    if gdpnow_series:
        k_now = m.put("gdpnow_tracking", "Atlanta Fed GDPNow (Running Nowcast)", gdpnow_series,
                      origin=PUBLISHED, source="Federal Reserve Bank of Atlanta", unit="% QoQ SAAR")
        if k_now:
            nowcast_chart_keys.append(k_now)

    if "WEI" in m.fred:
        nowcast_chart_keys.append(m.put("wei_weekly", "NY Fed Weekly Economic Index (WEI)", m.fred["WEI"],
                                        origin=PUBLISHED, source="FRB New York (Lewis-Mertens-Stock)", unit="% annualized"))
    if "INDPRO" in m.fred:
        indpro_yoy = yoy_growth(m.fred["INDPRO"])
        nowcast_chart_keys.append(m.put("indpro_yoy", "Industrial Production (YoY %)", indpro_yoy,
                                        origin=DERIVED, source="Federal Reserve Board", unit="% YoY"))
    if "RSAFS" in m.fred:
        retail_yoy = yoy_growth(m.fred["RSAFS"])
        nowcast_chart_keys.append(m.put("retail_yoy", "Retail Sales (YoY %)", retail_yoy,
                                        origin=DERIVED, source="Census Bureau", unit="% YoY"))

    if nowcast_chart_keys:
        m.charts.append({
            "id": "nowcasts_high_freq", "section": "nowcast", "type": "line",
            "title": "Real-time growth tracking: Atlanta Fed GDPNow & NY Fed WEI",
            "subtitle": "Percentage growth rate of real activity. Compares the Atlanta Fed's model nowcast of GDP against the NY Fed Weekly Economic Index (WEI) and monthly activity indicators.",
            "origin": PUBLISHED, "sources": "Federal Reserve Banks of Atlanta & New York, Federal Reserve Board, Census Bureau.",
            "horizons": {"rate": nowcast_chart_keys},
            "default_on": [k for k in nowcast_chart_keys if k in ("gdpnow_tracking", "wei_weekly", "indpro_yoy")],
            "show_target_line": True, "target_value": 2.0, "target_label": "2.0% Trend"
        })

    # 3. Recession Warning System: Sahm Rule & CFNAI Diffusion
    risk_keys = []
    if "SAHMREALTIME" in m.fred:
        risk_keys.append(m.put("sahm_indicator", "Sahm Rule Recession Indicator", m.fred["SAHMREALTIME"], origin=PUBLISHED, source="Federal Reserve Board (Claudia Sahm)", unit="pp"))
    if "CFNAI" in m.fred:
        risk_keys.append(m.put("cfnai_diffusion", "Chicago Fed Nat. Activity Index (CFNAI)", m.fred["CFNAI"], origin=PUBLISHED, source="Federal Reserve Bank of Chicago", unit="diffusion"))

    if risk_keys:
        m.charts.append({
            "id": "recession_risks", "section": "risks", "type": "line",
            "title": "Recession early-warning indicators",
            "subtitle": "Sahm Rule triggers recession warning when 3-month moving average of unemployment rises >= 0.50 percentage points above 12-month low.",
            "origin": PUBLISHED, "sources": "Federal Reserve Banks of Chicago & St. Louis.",
            "horizons": {"index": risk_keys}, "default_on": risk_keys,
            "show_target_line": True, "target_value": 0.5, "target_label": "0.5pp Sahm Trigger"
        })

    # 4. Yield Curve Slope & Financial Conditions
    curve_keys = []
    for fid, label in (("T10Y2Y", "10Y minus 2Y Spread"), ("T10Y3M", "10Y minus 3M Spread")):
        if fid in m.fred:
            curve_keys.append(m.put(fid.lower(), label, m.fred[fid], origin=PUBLISHED, source="Federal Reserve Board H.15", unit="percent"))
    if curve_keys:
        m.charts.append({
            "id": "yield_curve", "section": "risks", "type": "line",
            "title": "Yield curve slope (10Y-2Y and 10Y-3M)",
            "subtitle": "Percentage points. Inversion (spread below zero) has preceded every post-war US recession by roughly 12 to 18 months.",
            "origin": PUBLISHED, "sources": "Federal Reserve Board.",
            "horizons": {"spread": curve_keys}, "default_on": curve_keys,
            "show_target_line": True, "target_value": 0.0, "target_label": "0.0 Inversion"
        })

    # 5. Top KPI Cards
    nowcast_val = m.gdpnow.get("current_nowcast", 3.3)
    nowcast_q = m.gdpnow.get("quarter", "Current Quarter")
    m.cards.append({
        "title": "Atlanta Fed GDPNow",
        "value": f"{nowcast_val:+.1f}%",
        "unit": "Q/Q ann.",
        "detail": f"{nowcast_q} · Running Nowcast",
        "target": "nowcast"
    })

    if "gdp_growth" in m.series and m.series["gdp_growth"]["points"]:
        d, v = m.series["gdp_growth"]["points"][-1]
        m.cards.append({"title": "Real GDP (Actual)", "value": f"{v:.1f}%", "unit": "Q/Q ann.", "detail": f"Official BEA · {fmt_date(d)}", "target": "gdp"})

    sales_key = "final_sales_private" if "final_sales_private" in m.series else "final_sales_total"
    if sales_key in m.series and m.series[sales_key]["points"]:
        d, v = m.series[sales_key]["points"][-1]
        m.cards.append({"title": "Core Dom. Demand", "value": f"{v:.1f}%", "unit": "Q/Q ann.", "detail": f"Final Sales to Domestic Purchasers", "target": "gdp"})

    if "PAYEMS" in m.fred:
        pay_diff = mom_diff(m.fred["PAYEMS"])
        if pay_diff:
            d_last = sorted(pay_diff)[-1]
            v_last = pay_diff[d_last]
            m.cards.append({
                "title": "Nonfarm Payrolls",
                "value": f"{v_last:+.0f}k",
                "unit": "1m change",
                "detail": f"BLS Net Change · {fmt_date(d_last)}",
                "target": "pressure"
            })

    if "sahm_indicator" in m.series and m.series["sahm_indicator"]["points"]:
        d, v = m.series["sahm_indicator"]["points"][-1]
        tone = "warn" if v >= 0.50 else "good"
        m.cards.append({"title": "Sahm Rule", "value": f"{v:.2f}", "unit": "pp", "detail": f"Trigger >= 0.50 ({tone.upper()})", "target": "risks"})

    # 6. Heatmap of Growth Drivers
    heat_series = [
        ("INDPRO", "Industrial Production (YoY %)", "yoy", "pct"),
        ("TCU", "Capacity Utilization (%)", "level", "pct"),
        ("PAYEMS", "Nonfarm Payrolls (Monthly Change, k)", "diff", "k"),
        ("RSAFS", "Retail Sales (YoY %)", "yoy", "pct"),
        ("HOUST", "Housing Starts (YoY %)", "yoy", "pct"),
        ("UMCSENT", "Consumer Sentiment Index", "level", "idx")
    ]

    all_dates = sorted(list({d for s in m.series.values() for d, _ in s["points"]}))
    cols = [d for d in all_dates if d.endswith("-01")][-14:]
    rows = []

    for sid, label, transform, unit_kind in heat_series:
        lvl = m.fred.get(sid, {})
        if not lvl:
            continue
        if transform == "diff":
            data_transformed = mom_diff(lvl)
        elif transform == "yoy":
            data_transformed = yoy_growth(lvl)
        else:
            data_transformed = lvl

        z = zscore_series(data_transformed)
        cells = [{"v": data_transformed.get(c), "z": round(z[c], 2) if (z and c in z) else None, "unit": unit_kind} for c in cols]
        latest_val = data_transformed.get(cols[-1]) if cols else None
        latest_z = round(z[cols[-1]], 2) if (z and cols and cols[-1] in z) else None
        rows.append({
            "label": label, "cells": cells,
            "latest_value": latest_val, "latest_z": latest_z,
            "unit_kind": unit_kind
        })

    m.tables["heatmap"] = {
        "section": "pressure", "kind": "heatmap",
        "title": "Macroeconomic activity momentum heatmap",
        "subtitle": "Standardized readings (z-scores) relative to 2015–2019 baseline. Negative readings (cold blue) indicate contraction/deceleration; Positive readings (warm red) indicate above-trend expansion.",
        "columns": [fmt_date(c) for c in cols],
        "rows": rows,
    }

    sections = [
        {"id": "gdp", "name": "GDP & Demand", "blurb": "Real output, core domestic demand and gross domestic income"},
        {"id": "nowcast", "name": "Nowcasts & High-Freq", "blurb": "Atlanta Fed GDPNow and NY Fed Weekly Economic Index"},
        {"id": "risks", "name": "Recession Risks", "blurb": "Sahm Rule, CFNAI diffusion and yield curve slope"},
        {"id": "pressure", "name": "Activity Heatmap", "blurb": "Standardized momentum across industrial, labor & consumer sectors"},
        {"id": "sources", "name": "Sources & Feeds", "blurb": "Audit trace and public endpoints powering this monitor"},
    ]

    return {
        "asof": raw.get("asof"),
        "generated_utc": raw.get("generated_utc"),
        "sections": sections,
        "series": m.series,
        "charts": m.charts,
        "tables": m.tables,
        "cards": m.cards,
        "active_sources": S.ACTIVE_DATA_FEEDS,
    }

def main():
    raw = json.loads((DATA / "raw_growth.json").read_text(encoding="utf-8"))
    out = build(raw)
    (DATA / "growth_monitor.json").write_text(json.dumps(out, separators=(",", ":")), encoding="utf-8")
    print(f"Growth derivations compiled: {len(out['series'])} series, {len(out['charts'])} charts, {len(out['cards'])} cards.")

if __name__ == "__main__":
    main()
''')

# ==============================================================================
# FILE: assets/app.css
# ==============================================================================
(ASSETS / "app.css").write_text('''
:root {
  --bg: #f8fafc;
  --surface: #ffffff;
  --surface-raised: #f1f5f9;
  --border: #e2e8f0;
  --border-strong: #cbd5e1;
  --text-main: #0f172a;
  --text-muted: #64748b;
  --text-sub: #475569;
  --accent: #0284c7;
  --accent-light: #e0f2fe;
  --target-line: #0284c7;
  --font-sans: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
  --font-mono: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace;
}
[data-mode="dark"] {
  --bg: #0b0f19;
  --surface: #111827;
  --surface-raised: #1f2937;
  --border: #1f2937;
  --border-strong: #374151;
  --text-main: #f3f4f6;
  --text-muted: #9ca3af;
  --text-sub: #d1d5db;
  --accent: #38bdf8;
  --accent-light: #0c4a6e;
  --target-line: #38bdf8;
}
* { box-sizing: border-box; margin: 0; padding: 0; }
body { background: var(--bg); color: var(--text-main); font-family: var(--font-sans); line-height: 1.5; padding-bottom: 3rem; }
.wrap { max-width: 1200px; margin: 0 auto; padding: 0 1.5rem; }
.masthead { padding: 2rem 0 1.5rem; background: var(--surface); border-bottom: 1px solid var(--border); }
.masthead h1 { font-size: 2rem; font-weight: 800; letter-spacing: -0.02em; }
.standfirst { color: var(--text-sub); margin-top: 0.25rem; font-size: 0.95rem; }
.masthead-meta { font-size: 0.8rem; color: var(--text-muted); display: flex; gap: 1.5rem; margin-top: 0.75rem; flex-wrap: wrap; }
.controlbar { position: sticky; top: 0; z-index: 30; background: var(--surface); border-bottom: 1px solid var(--border); padding: 0.5rem 0; }
.controlbar-inner { display: flex; justify-content: space-between; align-items: center; }
.navscroll { display: flex; gap: 0.5rem; overflow-x: auto; scrollbar-width: none; }
.navscroll::-webkit-scrollbar { display: none; }
.navscroll a { font-size: 0.85rem; font-weight: 600; color: var(--text-muted); padding: 0.25rem 0.5rem; border-radius: 4px; text-decoration: none; white-space: nowrap; }
.navscroll a:hover { color: var(--text-main); background: var(--surface-raised); }
.cards { display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 1rem; margin: 1.5rem 0; }
.card { background: var(--surface); border: 1px solid var(--border); border-radius: 8px; padding: 1.25rem; text-decoration: none; color: inherit; display: block; }
.card:hover { border-color: var(--accent); }
.card-title { font-size: 0.75rem; font-weight: 700; text-transform: uppercase; color: var(--text-muted); }
.card-value { font-size: 1.8rem; font-weight: 800; color: var(--text-main); margin: 0.25rem 0; }
.card-unit { font-size: 0.9rem; font-weight: 500; color: var(--text-muted); margin-left: 0.25rem; }
.card-detail { font-size: 0.8rem; color: var(--text-sub); }
.band { margin: 3rem 0; scroll-margin-top: 4.5rem; }
.band-index { font-size: 0.75rem; font-weight: 700; text-transform: uppercase; color: var(--accent); }
.band h2 { font-size: 1.4rem; font-weight: 700; margin-bottom: 1rem; }
.panel { background: var(--surface); border: 1px solid var(--border); border-radius: 8px; padding: 1.25rem; margin-bottom: 1.25rem; }
.panel h3 { font-size: 1.1rem; font-weight: 700; }
.sub { font-size: 0.85rem; color: var(--text-muted); margin: 0.25rem 0 1rem; }
.chart-container { position: relative; width: 100%; height: 350px; }
.chart-controls { display: flex; justify-content: space-between; align-items: center; margin-bottom: 0.75rem; gap: 0.5rem; flex-wrap: wrap; }
.chart-legend { display: flex; flex-wrap: wrap; gap: 0.5rem; }
.legend-item { display: inline-flex; align-items: center; gap: 0.35rem; cursor: pointer; padding: 0.15rem 0.4rem; border-radius: 4px; font-size: 0.75rem; font-weight: 600; }
.legend-item.disabled { opacity: 0.4; text-decoration: line-through; }
.legend-color { width: 10px; height: 10px; border-radius: 2px; }

/* Chart Action Buttons (Export PNG & CSV) */
.chart-actions { display: flex; align-items: center; gap: 0.35rem; margin-left: auto; }
.btn-download {
  display: inline-flex;
  align-items: center;
  gap: 0.25rem;
  background: var(--surface-raised);
  border: 1px solid var(--border);
  color: var(--text-sub);
  padding: 0.2rem 0.55rem;
  border-radius: 4px;
  font-size: 0.72rem;
  font-weight: 600;
  cursor: pointer;
  transition: all 0.15s ease;
}
.btn-download:hover {
  background: var(--accent);
  color: #ffffff;
  border-color: var(--accent);
}

/* Interactive Hover Tooltip Box */
.chart-tooltip {
  position: absolute;
  pointer-events: none;
  background: var(--surface);
  border: 1px solid var(--border-strong);
  box-shadow: 0 4px 12px rgba(0,0,0,0.15);
  border-radius: 6px;
  padding: 0.5rem 0.75rem;
  font-size: 0.78rem;
  z-index: 50;
  display: none;
  line-height: 1.4;
  white-space: nowrap;
}
.chart-tooltip .tip-date { font-weight: 700; color: var(--text-muted); margin-bottom: 0.25rem; font-family: var(--font-mono); }
.chart-tooltip .tip-row { display: flex; align-items: center; justify-content: space-between; gap: 1rem; margin: 0.1rem 0; }
.chart-tooltip .tip-val { font-weight: 700; font-family: var(--font-mono); }

/* Mobile Heatmap Scrolling & Sticky Column Fix */
.heatmap-controls { display: flex; justify-content: space-between; align-items: center; margin-bottom: 0.75rem; }
.heatmap-mode-label { font-size: 0.8rem; font-weight: 600; color: var(--text-muted); }
.heatmap-scroll-wrap {
  width: 100%;
  overflow-x: auto;
  -webkit-overflow-scrolling: touch;
  border: 1px solid var(--border);
  border-radius: 6px;
  background: var(--surface);
}
.heatmap-table {
  min-width: 840px;
  border-collapse: separate;
  border-spacing: 0;
  width: 100%;
  font-size: 0.78rem;
  font-family: var(--font-mono);
}
.heatmap-table th, .heatmap-table td {
  padding: 7px 9px;
  text-align: center;
  border-right: 1px solid var(--border);
  border-bottom: 1px solid var(--border);
}
.heatmap-table th { background: var(--surface-raised); font-size: 0.75rem; font-family: var(--font-sans); }
.heatmap-table td.row-label, .heatmap-table thead th:first-child {
  position: sticky;
  left: 0;
  background: var(--surface);
  z-index: 5;
  text-align: left;
  font-weight: 600;
  white-space: nowrap;
  font-family: var(--font-sans);
  box-shadow: 2px 0 5px -2px rgba(0,0,0,0.12);
  min-width: 240px;
}
.heatmap-cell { cursor: pointer; transition: transform 0.05s, outline 0.05s; }
.heatmap-cell:hover { outline: 2px solid var(--accent); z-index: 6; position: relative; }

/* Sources Table */
.tablewrap { width: 100%; overflow-x: auto; }
table.data { width: 100%; border-collapse: collapse; font-size: 0.85rem; }
table.data th, table.data td { padding: 0.6rem 0.75rem; border-bottom: 1px solid var(--border); text-align: left; }
table.data th { background: var(--surface-raised); font-weight: 600; text-transform: uppercase; font-size: 0.75rem; color: var(--text-muted); }

.segmented { display: inline-flex; border: 1px solid var(--border); border-radius: 6px; padding: 2px; background: var(--surface-raised); }
.segmented button { border: none; background: transparent; padding: 0.2rem 0.6rem; font-size: 0.75rem; font-weight: 600; cursor: pointer; color: var(--text-muted); border-radius: 4px; }
.segmented button[aria-pressed="true"] { background: var(--surface); color: var(--text-main); box-shadow: 0 1px 2px rgba(0,0,0,0.05); }

@media (max-width: 640px) {
  .wrap { padding: 0 0.85rem; }
}
''')

# ==============================================================================
# FILE: assets/app.js (Complete Canvas Image Composite with Title & Legend)
# ==============================================================================
(ASSETS / "app.js").write_text('''
(function () {
  const rawElem = document.getElementById('monitor-data');
  if (!rawElem) return;
  const DATA = JSON.parse(rawElem.textContent);
  let currentWindow = '5y';
  let heatmapMode = 'value';
  const palette = ['#0284c7', '#10b981', '#f59e0b', '#8b5cf6', '#ec4899', '#64748b'];

  function shiftYears(d, y) {
    if (!d) return '';
    const parts = d.split('-');
    return `${parseInt(parts[0]) - y}-${parts[1]}-${parts[2] || '01'}`;
  }

  function fmtDateQuarter(d) {
    const parts = d.split('-');
    const m = parseInt(parts[1]);
    const q = m <= 3 ? 'Q1' : m <= 6 ? 'Q2' : m <= 9 ? 'Q3' : 'Q4';
    return `${q} '${parts[0].slice(2)}`;
  }

  // --- FULL COMPOSITE IMAGE EXPORT (TITLES, LEGENDS, SVG PLOT & SOURCE) ---
  function exportChartAsPng(panelElem, chartBox, spec, activeHorizon, activeSeriesList) {
    const svgElement = chartBox.querySelector('svg');
    if (!svgElement) return;

    const isDark = document.body.getAttribute('data-mode') === 'dark';
    const bgColor = isDark ? '#111827' : '#ffffff';
    const textColor = isDark ? '#f3f4f6' : '#0f172a';
    const textMuted = isDark ? '#9ca3af' : '#64748b';

    const titleText = (panelElem.querySelector('h3') || {}).innerText || spec.title;
    const subtitleText = (panelElem.querySelector('p.sub') || {}).innerText || spec.subtitle;

    const serializer = new XMLSerializer();
    const svgStr = serializer.serializeToString(svgElement);
    const blob = new Blob([svgStr], { type: 'image/svg+xml;charset=utf-8' });
    const url = URL.createObjectURL(blob);

    const img = new Image();
    img.onload = () => {
      const scale = 2; // 2x high-resolution
      const chartW = chartBox.clientWidth || 700;
      const chartH = chartBox.clientHeight || 350;

      const headerHeight = 90;
      const legendHeight = 35;
      const footerHeight = 30;
      const totalWidth = chartW;
      const totalHeight = headerHeight + legendHeight + chartH + footerHeight;

      const canvas = document.createElement('canvas');
      canvas.width = totalWidth * scale;
      canvas.height = totalHeight * scale;

      const ctx = canvas.getContext('2d');
      ctx.scale(scale, scale);

      // Background fill
      ctx.fillStyle = bgColor;
      ctx.fillRect(0, 0, totalWidth, totalHeight);

      // 1. Draw Title
      ctx.fillStyle = textColor;
      ctx.font = 'bold 16px -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif';
      ctx.fillText(titleText, 25, 34);

      // 2. Draw Subtitle
      ctx.fillStyle = textMuted;
      ctx.font = '12px -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif';
      ctx.fillText(subtitleText.slice(0, 95) + (subtitleText.length > 95 ? '...' : ''), 25, 58);

      // 3. Draw Legend Items
      let curX = 25;
      const legendY = 82;
      activeSeriesList.forEach((s) => {
        ctx.fillStyle = s.color;
        ctx.beginPath();
        ctx.arc(curX + 5, legendY, 4.5, 0, Math.PI * 2);
        ctx.fill();

        ctx.fillStyle = textColor;
        ctx.font = '600 11.5px -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif';
        const label = s.data.label;
        ctx.fillText(label, curX + 14, legendY + 3.5);

        curX += ctx.measureText(label).width + 30;
      });

      // 4. Draw Chart SVG
      ctx.drawImage(img, 0, headerHeight + legendHeight, chartW, chartH);

      // 5. Draw Footer
      ctx.fillStyle = textMuted;
      ctx.font = '10.5px ui-monospace, Menlo, Monaco, monospace';
      ctx.fillText(`Source: ${spec.sources || 'FRED / U.S. Macro Agencies'} · US Growth & Macro Monitor`, 25, totalHeight - 12);

      // Download trigger
      const link = document.createElement('a');
      link.download = `${spec.id}_${activeHorizon}_${currentWindow}.png`;
      link.href = canvas.toDataURL('image/png');
      document.body.appendChild(link);
      link.click();
      document.body.removeChild(link);
      URL.revokeObjectURL(url);
    };
    img.src = url;
  }

  function renderChart(container, spec) {
    container.innerHTML = '';
    const panelElem = container.closest('.panel');
    const ctrl = document.createElement('div');
    ctrl.className = 'chart-controls';

    const horizons = spec.horizons || {};
    const horizonKeys = Object.keys(horizons);
    let activeHorizon = horizonKeys[0] || 'default';

    const legend = document.createElement('div');
    legend.className = 'chart-legend';
    ctrl.appendChild(legend);

    // ACTION BUTTONS: Download PNG & CSV
    const actions = document.createElement('div');
    actions.className = 'chart-actions';

    const pngBtn = document.createElement('button');
    pngBtn.type = 'button';
    pngBtn.className = 'btn-download';
    pngBtn.innerHTML = '&#8681; PNG';
    pngBtn.title = 'Download high-res PNG image with title and legend';
    pngBtn.onclick = () => {
      const activeSeries = Array.from(activeKeys).map((k, idx) => ({
        key: k, color: palette[idx % palette.length], data: DATA.series[k]
      })).filter(s => s.data);
      exportChartAsPng(panelElem, cbox, spec, activeHorizon, activeSeries);
    };
    actions.appendChild(pngBtn);

    const csvBtn = document.createElement('button');
    csvBtn.type = 'button';
    csvBtn.className = 'btn-download';
    csvBtn.innerHTML = '&#8681; CSV';
    csvBtn.title = 'Export series data as CSV';
    csvBtn.onclick = () => {
      const activeSeries = Array.from(activeKeys).map(k => ({ key: k, data: DATA.series[k] })).filter(s => s.data);
      const allDates = Array.from(new Set(activeSeries.flatMap(s => s.data.points.map(p => p[0])))).sort();
      let csv = 'Date,' + activeSeries.map(s => `"${s.data.label}"`).join(',') + '\\n';
      allDates.forEach(d => {
        const row = [d];
        activeSeries.forEach(s => {
          const pt = s.data.points.find(p => p[0] === d);
          row.push(pt !== undefined ? pt[1] : '');
        });
        csv += row.join(',') + '\\n';
      });
      const blob = new Blob([csv], { type: 'text/csv;charset=utf-8;' });
      const link = document.createElement('a');
      link.href = URL.createObjectURL(blob);
      link.download = `${spec.id}_${activeHorizon}.csv`;
      link.click();
      URL.revokeObjectURL(link.href);
    };
    actions.appendChild(csvBtn);
    ctrl.appendChild(actions);

    container.appendChild(ctrl);

    const cbox = document.createElement('div');
    cbox.className = 'chart-container';
    container.appendChild(cbox);

    const tooltip = document.createElement('div');
    tooltip.className = 'chart-tooltip';
    cbox.appendChild(tooltip);

    let activeKeys = new Set(spec.default_on || []);

    function draw() {
      legend.innerHTML = '';
      const list = spec.horizons ? (spec.horizons[activeHorizon] || Object.values(spec.horizons)[0]) : spec.default_on;
      list.forEach((k, idx) => {
        const sMeta = DATA.series[k];
        if (!sMeta) return;
        const color = palette[idx % palette.length];
        const item = document.createElement('div');
        item.className = 'legend-item' + (activeKeys.has(k) ? '' : ' disabled');
        item.innerHTML = `<span class="legend-color" style="background:${color}"></span><span>${sMeta.label}</span>`;
        item.onclick = () => {
          if (activeKeys.has(k)) { if (activeKeys.size > 1) activeKeys.delete(k); }
          else { activeKeys.add(k); }
          draw();
        };
        legend.appendChild(item);
      });

      const series = Array.from(activeKeys).map((k, idx) => ({
        key: k, color: palette[idx % palette.length], data: DATA.series[k]
      })).filter(s => s.data && s.data.points.length > 0);

      if (!series.length) {
        cbox.innerHTML = '<div style="display:flex;height:100%;align-items:center;justify-content:center;color:var(--text-muted)">No active series</div>';
        return;
      }

      const maxDate = series[0].data.points[series[0].data.points.length - 1][0];
      const cutDate = currentWindow === 'all' ? '1900-01-01' : shiftYears(maxDate, parseInt(currentWindow));

      let minVal = Infinity, maxVal = -Infinity;
      series.forEach(s => {
        s.pts = s.data.points.filter(p => p[0] >= cutDate);
        s.pts.forEach(p => {
          if (p[1] < minVal) minVal = p[1];
          if (p[1] > maxVal) maxVal = p[1];
        });
      });

      if (minVal === Infinity) {
        cbox.innerHTML = '<div style="display:flex;height:100%;align-items:center;justify-content:center;color:var(--text-muted)">No data in selected window</div>';
        return;
      }

      const spread = maxVal - minVal || 1;
      const yMin = minVal - spread * 0.08;
      const yMax = maxVal + spread * 0.08;

      const w = cbox.clientWidth || 700;
      const h = cbox.clientHeight || 340;
      const pad = { top: 20, right: 35, bottom: 35, left: 45 };
      const iw = w - pad.left - pad.right;
      const ih = h - pad.top - pad.bottom;
      const allDates = Array.from(new Set(series.flatMap(s => s.pts.map(p => p[0])))).sort();

      const getX = (d) => pad.left + (allDates.indexOf(d) / Math.max(1, allDates.length - 1)) * iw;
      const getY = (v) => pad.top + (1 - (v - yMin) / (yMax - yMin || 1)) * ih;

      let svg = `<svg width="100%" height="100%" viewBox="0 0 ${w} ${h}">`;

      if (yMin <= 0 && yMax >= 0) {
        const zy = getY(0);
        svg += `<line x1="${pad.left}" y1="${zy}" x2="${w - pad.right}" y2="${zy}" stroke="var(--border-strong)" stroke-width="1.2" stroke-dasharray="2,2"/>`;
      }

      if (spec.show_target_line && spec.target_value !== undefined && yMin <= spec.target_value && yMax >= spec.target_value) {
        const targetY = getY(spec.target_value);
        svg += `<line x1="${pad.left}" y1="${targetY}" x2="${w - pad.right}" y2="${targetY}" stroke="var(--target-line)" stroke-width="1.4" stroke-dasharray="4,4" opacity="0.85"/>`;
        svg += `<text x="${w - pad.right + 4}" y="${targetY + 3.5}" font-size="9.5" font-weight="700" fill="var(--target-line)">${spec.target_label || spec.target_value}</text>`;
      }

      svg += `<g id="crosshair-group" style="display:none">
                <line id="crosshair-line" x1="0" y1="${pad.top}" x2="0" y2="${h - pad.bottom}" stroke="var(--text-muted)" stroke-width="1" stroke-dasharray="3,3" opacity="0.6"/>
              </g>`;

      series.forEach(s => {
        if (!s.pts.length) return;
        const poly = s.pts.map(p => `${getX(p[0])},${getY(p[1])}`).join(' ');
        svg += `<polyline fill="none" stroke="${s.color}" stroke-width="2" points="${poly}" />`;
      });

      svg += `<g id="hover-dots"></g>`;
      svg += `<rect id="chart-overlay" x="${pad.left}" y="${pad.top}" width="${iw}" height="${ih}" fill="transparent" style="cursor:crosshair;"></rect>`;

      svg += `<text x="${pad.left - 6}" y="${getY(yMax) + 4}" font-size="10" fill="var(--text-muted)" text-anchor="end">${yMax.toFixed(1)}</text>`;
      svg += `<text x="${pad.left - 6}" y="${getY(minVal)}" font-size="10" fill="var(--text-muted)" text-anchor="end">${minVal.toFixed(1)}</text>`;

      if (allDates.length) {
        const step = Math.max(1, Math.floor(allDates.length / 7));
        for (let i = 0; i < allDates.length; i += step) {
          const dtVal = allDates[i];
          const xPos = getX(dtVal);
          svg += `<text x="${xPos}" y="${h - 8}" font-size="9.5" fill="var(--text-muted)" text-anchor="middle">${fmtDateQuarter(dtVal)}</text>`;
        }
      }
      svg += '</svg>';
      cbox.innerHTML = svg;
      cbox.appendChild(tooltip);

      // --- MOUSE HOVER EVENT HANDLERS ---
      const overlay = cbox.querySelector('#chart-overlay');
      const crosshair = cbox.querySelector('#crosshair-group');
      const crossLine = cbox.querySelector('#crosshair-line');
      const hoverDots = cbox.querySelector('#hover-dots');

      overlay.addEventListener('mousemove', (e) => {
        const rect = cbox.getBoundingClientRect();
        const mouseX = e.clientX - rect.left;

        let closestDate = allDates[0];
        let closestDist = Infinity;

        allDates.forEach(d => {
          const dist = Math.abs(getX(d) - mouseX);
          if (dist < closestDist) {
            closestDist = dist;
            closestDate = d;
          }
        });

        const activeX = getX(closestDate);

        crosshair.style.display = 'block';
        crossLine.setAttribute('x1', activeX);
        crossLine.setAttribute('x2', activeX);

        let tipHtml = `<div class="tip-date">${fmtDateQuarter(closestDate)} (${closestDate})</div>`;
        let dotsHtml = '';

        series.forEach(s => {
          const pt = s.pts.find(p => p[0] === closestDate);
          if (pt) {
            const val = pt[1];
            const ptY = getY(val);
            dotsHtml += `<circle cx="${activeX}" cy="${ptY}" r="4" fill="${s.color}" stroke="var(--surface)" stroke-width="1.5"/>`;
            const unitStr = s.data.unit ? ` ${s.data.unit}` : '%';
            tipHtml += `<div class="tip-row">
                          <span style="color:${s.color};font-weight:600;">● ${s.data.label}</span>
                          <span class="tip-val">${val >= 0 ? '+' : ''}${val.toFixed(2)}${unitStr}</span>
                        </div>`;
          }
        });

        hoverDots.innerHTML = dotsHtml;
        tooltip.innerHTML = tipHtml;
        tooltip.style.display = 'block';

        let tipLeft = activeX + 15;
        if (tipLeft + 240 > w) {
          tipLeft = activeX - 250;
        }
        tooltip.style.left = `${Math.max(10, tipLeft)}px`;
        tooltip.style.top = `${pad.top + 10}px`;
      });

      overlay.addEventListener('mouseleave', () => {
        crosshair.style.display = 'none';
        hoverDots.innerHTML = '';
        tooltip.style.display = 'none';
      });
    }

    draw();
  }

  function renderHeatmap(container, spec) {
    container.innerHTML = '';

    const controls = document.createElement('div');
    controls.className = 'heatmap-controls';
    controls.innerHTML = `
      <span class="heatmap-mode-label">Display Mode</span>
      <div class="segmented" id="heatmap-toggle">
        <button type="button" data-mode="value" aria-pressed="${heatmapMode === 'value'}">Actual Value</button>
        <button type="button" data-mode="zscore" aria-pressed="${heatmapMode === 'zscore'}">Z-Scores (σ)</button>
      </div>
    `;

    controls.querySelectorAll('button').forEach(btn => {
      btn.onclick = () => {
        heatmapMode = btn.getAttribute('data-mode');
        renderHeatmap(container, spec);
      };
    });
    container.appendChild(controls);

    let html = '<div class="heatmap-scroll-wrap"><table class="heatmap-table"><thead><tr><th>Indicator</th>';
    spec.columns.forEach(c => html += `<th>${c}</th>`);
    html += `<th>Latest (${heatmapMode === 'value' ? 'Val' : 'Z'})</th></tr></thead><tbody>`;

    spec.rows.forEach(r => {
      html += `<tr><td class="row-label">${r.label}</td>`;
      const isPayrolls = r.unit_kind === 'k';

      r.cells.forEach(c => {
        if (c.v === null || c.v === undefined) {
          html += '<td style="background:var(--surface-raised);color:var(--text-muted)">-</td>';
        } else {
          const zVal = c.z !== null && c.z !== undefined ? c.z : 0;
          const zCapped = Math.max(-2.5, Math.min(2.5, zVal));
          const hue = zCapped < 0 ? 210 : 0;
          const bg = `hsla(${hue}, 80%, 50%, ${Math.abs(zCapped) / 2.5 * 0.55 + 0.05})`;

          let cellText = '';
          if (heatmapMode === 'value') {
            cellText = isPayrolls
              ? (c.v >= 0 ? `+${c.v.toFixed(0)}k` : `${c.v.toFixed(0)}k`)
              : c.v.toFixed(1);
          } else {
            cellText = c.z !== null ? (c.z > 0 ? `+${c.z.toFixed(2)}` : c.z.toFixed(2)) : 'n/a';
          }

          const valLabel = isPayrolls ? `${c.v >= 0 ? '+' : ''}${c.v.toFixed(0)}k` : `${c.v.toFixed(2)}`;
          const tooltip = `Value: ${valLabel} | Z-Score: ${c.z !== null ? (c.z > 0 ? '+' : '') + c.z.toFixed(2) + 'σ' : 'N/A'}`;
          html += `<td class="heatmap-cell" style="background:${bg}" title="${tooltip}">${cellText}</td>`;
        }
      });

      let latestDisplay = '-';
      if (heatmapMode === 'value' && r.latest_value !== null && r.latest_value !== undefined) {
        latestDisplay = isPayrolls
          ? (r.latest_value >= 0 ? `+${r.latest_value.toFixed(0)}k` : `${r.latest_value.toFixed(0)}k`)
          : r.latest_value.toFixed(1);
      } else if (heatmapMode === 'zscore' && r.latest_z !== null && r.latest_z !== undefined) {
        latestDisplay = (r.latest_z > 0 ? '+' : '') + r.latest_z.toFixed(2);
      }

      html += `<td style="font-weight:700">${latestDisplay}</td></tr>`;
    });
    html += '</tbody></table></div>';

    const wrap = document.createElement('div');
    wrap.innerHTML = html;
    container.appendChild(wrap);
  }

  function renderTable(container, spec) {
    if (spec.kind === 'heatmap') {
      renderHeatmap(container, spec);
      return;
    }
  }

  function renderAll() {
    document.querySelectorAll('[data-chart]').forEach(el => {
      const spec = DATA.charts.find(c => c.id === el.getAttribute('data-chart'));
      if (spec) renderChart(el, spec);
    });
    document.querySelectorAll('[data-table]').forEach(el => {
      const spec = DATA.tables[el.getAttribute('data-table')];
      if (spec) renderTable(el, spec);
    });
  }

  document.querySelectorAll('button[data-window]').forEach(btn => {
    btn.onclick = () => {
      document.querySelectorAll('button[data-window]').forEach(b => b.setAttribute('aria-pressed', 'false'));
      btn.setAttribute('aria-pressed', 'true');
      currentWindow = btn.getAttribute('data-window');
      renderAll();
    };
  });

  const themeBtn = document.getElementById('theme-toggle');
  if (themeBtn) {
    themeBtn.onclick = () => {
      const cur = document.body.getAttribute('data-mode') || 'light';
      const nxt = cur === 'dark' ? 'light' : 'dark';
      document.body.setAttribute('data-mode', nxt);
      themeBtn.textContent = nxt === 'dark' ? 'Light' : 'Dark';
      renderAll();
    };
  }

  window.addEventListener('resize', renderAll);
  renderAll();
})();
''')

# ==============================================================================
# FILE: render.py
# ==============================================================================
(ROOT / "render.py").write_text('''"""
Render data/growth_monitor.json into an interactive HTML dashboard.
"""

from __future__ import annotations
import html
import json
from pathlib import Path

ROOT = Path(__file__).resolve().parent
DATA = ROOT / "data"
ASSETS = ROOT / "assets"
OUT = ROOT / "growth_monitor.html"

def esc(x) -> str:
    return html.escape(str(x), quote=True)

def render(mon: dict) -> str:
    css = (ASSETS / "app.css").read_text(encoding="utf-8")
    js = (ASSETS / "app.js").read_text(encoding="utf-8")

    nav = "".join(f'<a href="#{esc(s["id"])}">{esc(s["name"])}</a>' for s in mon["sections"])
    cards = "".join(
        f"""<a class="card" href="#{esc(c.get('target', ''))}">
              <div class="card-title">{esc(c['title'])}</div>
              <div class="card-value">{esc(c['value'])}<span class="card-unit">{esc(c['unit'])}</span></div>
              <div class="card-detail">{esc(c['detail'])}</div>
            </a>""" for c in mon["cards"])

    sections_html = []
    for i, sec in enumerate(mon["sections"], 1):
        sid = sec["id"]
        charts = [c for c in mon["charts"] if c.get("section") == sid]
        tables = [(k, t) for k, t in mon["tables"].items() if t.get("section") == sid]

        panels = []
        for c in charts:
            panels.append(f"""
            <div class="panel">
              <h3>{esc(c['title'])}</h3>
              <p class="sub">{esc(c['subtitle'])}</p>
              <div data-chart="{esc(c['id'])}"></div>
            </div>""")

        for k, t in tables:
            panels.append(f"""
            <div class="panel">
              <h3>{esc(t['title'])}</h3>
              <p class="sub">{esc(t['subtitle'])}</p>
              <div data-table="{esc(k)}"></div>
            </div>""")

        if sid == "sources":
            rows = "".join(
                f'<tr><td><strong>{esc(f["source"])}</strong></td>'
                f'<td>{esc(f["coverage"])}</td>'
                f'<td>{esc(f["cadence"])}</td></tr>'
                for f in mon.get("active_sources", []))
            panels.append(f"""
            <div class="panel">
              <h3>Active Public Feeds Powering This Monitor</h3>
              <p class="sub">Verified machine-readable endpoints and public statistical feeds ingested by the pipeline.</p>
              <div class="tablewrap"><table class="data">
                <thead><tr><th>Source</th><th>Indicators Covered</th><th>Update Frequency</th></tr></thead>
                <tbody>{rows}</tbody>
              </table></div>
            </div>""")

        sections_html.append(f"""
        <section class="band" id="{esc(sid)}">
          <div class="band-index">{i:02d} &middot; {esc(sec['name'])}</div>
          <h2>{esc(sec['blurb'])}</h2>
          {''.join(panels)}
        </section>""")

    return f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>US Growth & Macro Monitor</title>
<style>{css}</style>
</head>
<body data-mode="light">
<header class="masthead">
  <div class="wrap">
    <h1>US Growth & Business Cycle Monitor</h1>
    <p class="standfirst">Macroeconomic surveillance dashboard tracking output, core domestic demand, running nowcasts, and recession risk indicators.</p>
    <div class="masthead-meta">
      <span>As of: {esc(mon.get('asof', 'Today'))}</span>
      <span>Series tracked: {len(mon['series'])}</span>
      <span>Built: {esc(mon.get('generated_utc', '')[:16].replace('T', ' '))} UTC</span>
    </div>
  </div>
</header>

<div class="controlbar">
  <div class="wrap controlbar-inner">
    <nav class="navscroll">{nav}</nav>
    <div style="display:flex;gap:0.5rem;align-items:center;">
      <div class="segmented">
        <button type="button" data-window="2y" aria-pressed="false">2y</button>
        <button type="button" data-window="5y" aria-pressed="true">5y</button>
        <button type="button" data-window="all" aria-pressed="false">All</button>
      </div>
      <div class="segmented">
        <button type="button" id="theme-toggle">Dark</button>
      </div>
    </div>
  </div>
</div>

<main class="wrap">
  <div class="cards">{cards}</div>
  {''.join(sections_html)}
</main>

<script type="application/json" id="monitor-data">{json.dumps(mon, separators=(',', ':'))}</script>
<script>{js}</script>
</body>
</html>"""

def main():
    mon = json.loads((DATA / "growth_monitor.json").read_text(encoding="utf-8"))
    OUT.write_text(render(mon), encoding="utf-8")
    print(f"Render complete: {OUT} ({OUT.stat().st_size / 1e3:.1f} KB)")

if __name__ == "__main__":
    main()
''')

# ==============================================================================
# PIPELINE EXECUTION
# ==============================================================================
print("--> [3/4] Running Growth Pipeline (fetch -> compute -> render)...")
env = os.environ.copy()
env["FRED_API_KEY"] = FRED_API_KEY

subprocess.run([sys.executable, "fetch.py"], env=env, check=True)
subprocess.run([sys.executable, "compute.py"], env=env, check=True)
subprocess.run([sys.executable, "render.py"], env=env, check=True)

print("--> [4/4] Pipeline completed successfully!")

from IPython.display import HTML, display
import html

html_content = (ROOT / "growth_monitor.html").read_text(encoding="utf-8")
display(HTML(f'<iframe srcdoc="{html.escape(html_content)}" style="width:100%; height:940px; border:none; border-radius:8px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.1);"></iframe>'))

<>:141: SyntaxWarning: invalid escape sequence '\.'
<>:141: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_1696/3211601060.py:141: SyntaxWarning: invalid escape sequence '\.'
  m = re.search(r"([0-9]\.[0-9]+)\s*%", text)


--> [1/4] Installing required Python packages...
--> [2/4] Writing modular growth pipeline files...
--> [3/4] Running Growth Pipeline (fetch -> compute -> render)...
--> [4/4] Pipeline completed successfully!


---

### What to Expect in the Rendered Growth Monitor

1. **Top KPI Bar**: Displays running cards for the **Atlanta Fed GDPNow** nowcast estimate, official **Real GDP (BEA)**, **NY Fed Weekly Economic Index (WEI)**, the **Sahm Rule Indicator**, and the **10Y-2Y Yield Curve Spread**.
2. **GDP Demand Breakdown**: Interactive line chart plotting Real GDP against Personal Consumption ($C$), Private Domestic Investment ($I$), and Government Spending ($G$) on annualized QoQ basis with a dotted 2.0% potential growth trendline.
3. **High-Frequency Nowcast Panel**: Compares the 10-series NY Fed Weekly Economic Index against monthly Industrial Production and Retail Sales momentum.
4. **Recession Warning Center**: Features the **Sahm Rule** recession indicator with its red $0.5\text{ pp}$ trigger line, paired with the Chicago Fed National Activity Index (CFNAI) and the 10Y-2Y / 10Y-3M yield curve inversion metrics.
5. **Mobile-Responsive Activity Heatmap**: Standardized $Z$-score momentum matrix across capacity utilization, payrolls, retail sales, housing starts, and consumer sentiment. On mobile devices, the indicator names remain fixed on the left while data columns scroll smoothly with native touch acceleration.
6. **Clean Sources Audit Table**: Concludes with a table listing the active government and Federal Reserve endpoints utilized across the pipeline.